# Capítulo 13: Regressão Logística

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 16 de Grus (2019).

> Muita gente diz que existe uma linha tênue entre a genialidade e a loucura. Eu não acho que a linha seja tênue — acho que existe um abismo entre as duas.
>
> — Bill Bailey

O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) terminou com um modelo que prevê **quanto**: minutos por dia, um número contínuo, com um coeficiente por variável explicativa. Este capítulo troca a pergunta por outra, que na prática é a mais comum das duas: **sim ou não**. O usuário pagou pela conta premium? A mensagem é spam? O candidato passa na entrevista? A variável a prever deixa de ser uma quantidade e passa a ser um rótulo, codificado como 0 ou 1.

A [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) faz a coisa óbvia: pega o modelo linear do capítulo anterior, sem alterar uma linha, e o aponta para um alvo 0/1. Ele roda, devolve coeficientes e prevê — números negativos, e números maiores que 1, para uma variável que só assume os valores 0 e 1. É um fracasso instrutivo, e ele define o problema que o resto do capítulo resolve: a saída precisa ficar presa no intervalo $[0, 1]$ para poder ser lida como probabilidade.

Este capítulo também cobra duas dívidas antigas. A primeira é da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html), que prometeu que a regressão logística **não tem fórmula fechada** — não existem duas médias e uma covariância que resolvam este problema, e nem sequer um sistema linear a resolver, como havia na regressão múltipla. O gradiente descendente deixa de ser o caminho mais prático entre dois possíveis e passa a ser o **único** caminho: sem ele, não há modelo. A segunda dívida é da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), que introduziu estimação por máxima verossimilhança e mostrou que, para erros normais, maximizar a verossimilhança é exatamente minimizar a soma dos quadrados. Aqui os erros não são normais — o alvo é binário —, aquela equivalência some, e sobra a verossimilhança sozinha. É ela que vamos otimizar, na forma em que um computador consegue: **minimizando a log-verossimilhança negativa**.

E há um terceiro fio, que só se enxerga na última seção. Ajustar $\beta$ produz, de brinde, um **hiperplano** que separa as duas classes — o conjunto de pontos onde `x @ beta` vale zero. Encontrar diretamente o hiperplano que melhor separa as classes, sem passar por probabilidade nenhuma, é um critério diferente e dá origem a outra família de classificadores: as **máquinas de vetores de suporte**. A [seção 13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) fecha o capítulo comparando os dois critérios.

Ao final deste capítulo, você será capaz de:

- Explicar por que um modelo linear não serve para prever uma variável binária, e demonstrar isso com previsões fora do intervalo $[0, 1]$
- Escrever a função logística, justificar por que ela resolve o problema do alcance e derivar a sua derivada
- Derivar a log-verossimilhança de um modelo com alvo binário e explicar por que maximizá-la equivale a minimizar a log-verossimilhança negativa
- Ajustar uma regressão logística por gradiente descendente, sem nenhuma fórmula fechada disponível
- Explicar por que um ajuste pode reportar uma perda impossível — `nan` — e ainda assim rodar até o fim, e por que nenhuma biblioteca teria mostrado isso a você
- Avaliar um classificador probabilístico com precisão e revocação, e explicar de onde vem o limiar de 0,5
- Descrever o critério de margem máxima de uma máquina de vetores de suporte e dizer em que ele difere de maximizar a verossimilhança

## Seções

| Seção | Tópico |
|---|---|
| [13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) | O Problema |
| [13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) | A Função Logística |
| [13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) | Aplicando o Modelo |
| [13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html) | Qualidade do Ajuste |
| [13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) | Máquinas de Vetores de Suporte |

## O Problema

> **📌 Nota**
>
> Esta seção corresponde a *The Problem*, do capítulo 16 de Grus (2019).

O [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html) já encostou neste problema. Lá, com um punhado de usuários e nenhuma técnica, olhamos anos de experiência contra conta paga, vimos que os extremos pagavam e o meio não, e escrevemos um classificador com os cortes chutados à mão — `if anos < 3.0 ... elif anos < 8.5 ...`. Aquele código foi apresentado como o que era: um modelo que lê os cortes dos próprios dados que deveria explicar, e que não sobrevive ao primeiro usuário novo.

Agora temos ferramentas. Vamos refazer o problema direito.

### Os dados

O conjunto tem cerca de 200 usuários anonimizados, e para cada um sabemos três coisas: **anos de experiência** como cientista de dados, **salário**, e se a pessoa **pagou** pela conta premium. Como é típico com variáveis categóricas, o alvo é representado como 0 (não pagou) ou 1 (pagou) — a mesma variável indicadora que a [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) usou para "tem doutorado", agora do lado de fora do modelo, como a coisa a prever.

Cada linha bruta é `[experiência, salário, conta_paga]`. Convertendo para o formato de que precisamos — com a coluna de 1 na frente, pela convenção do capítulo anterior:

```python
xs = np.column_stack([np.ones(len(data)), data[:, :2]])   # [1, experiência, salário]
ys = data[:, 2]                                           # conta_paga
```

`np.column_stack` cola vetores como colunas de uma matriz; o resultado é o `xs` de forma `(200, 3)` que `scratch_np.logistic_regression` exporta, junto com o `ys` de forma `(200,)`.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from scratch_np.logistic_regression import xs, ys

In [ ]:
xs.shape, xs[0], ys[0], ys.sum()

São 200 usuários, dos quais 52 pagaram — 26% da base. A primeira linha é um usuário com 0,7 ano de experiência, salário de 48.000 e conta paga.

Vale olhar os dados antes de modelar. `ys == 1` é uma máscara booleana, e `xs[paga, 1]` seleciona a coluna da experiência só nas linhas que ela marca — o `~paga` faz o complemento. É a máscara da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) com o `~` da [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html), no lugar dos quatro `zip` que a separação por classe exigiria de outro jeito.

In [ ]:
# Figura: Usuários pagantes e não pagantes
paga = ys == 1

plt.scatter(xs[paga, 1], xs[paga, 2], marker='+', label='paga')
plt.scatter(xs[~paga, 1], xs[~paga, 2], marker='.', label='não paga')
plt.xlabel("anos de experiência")
plt.ylabel("salário")
plt.legend(loc='upper left')
plt.title("Usuários pagantes e não pagantes")
plt.show()

Os pagantes ficam quase todos **abaixo** da nuvem principal, e sobretudo à direita dela: **muita experiência e salário relativamente baixo**. Faz algum sentido — quem tem experiência e ainda não está ganhando bem talvez esteja procurando emprego, que é para o que serve uma conta premium numa rede de cientistas de dados. Há também um punhado de pagantes no canto inferior esquerdo, com pouca experiência e os salários mais baixos do conjunto.

Guarde essa observação; ela vai reaparecer como um coeficiente negativo na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html), e o sinal negativo assusta quem não olhou o gráfico antes.

### A tentativa óbvia

A primeira tentativa óbvia é usar regressão linear e achar o melhor modelo:

$$
\text{conta paga} = \beta_0 + \beta_1 \,\text{experiência} + \beta_2 \,\text{salário} + \varepsilon
$$

Nada impede de modelar o problema assim. Literalmente nada: a função `least_squares_fit` do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) aceita qualquer array `ys` de forma `(n,)`, e não tem como saber que estes só valem 0 e 1.

Antes de ajustar, um passo de higiene: **reescalonar**. Experiência vai de 0,1 a 10; salário vai de 30.000 a 107.000. Com amplitudes que diferem por um fator de quase 8.000, a coluna do salário domina qualquer produto escalar, e o gradiente descendente com taxa de aprendizado única não tem chance. A função é a mesma que a [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) construiu, repetida aqui porque cada página deste livro roda num kernel próprio — `axis=0` reduz coluna a coluna, `ddof=1` é o desvio amostral, e `varia` é a máscara que protege as colunas sem variação:

In [ ]:
from typing import Tuple

def scale(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """devolve a média e o desvio padrão de cada coluna"""
    X = np.asarray(X, dtype=float)
    return X.mean(axis=0), X.std(axis=0, ddof=1)

def rescale(X: np.ndarray) -> np.ndarray:
    """cada coluna passa a ter média 0 e desvio padrão 1"""
    X = np.asarray(X, dtype=float)
    means, stdevs = scale(X)

    rescaled = X.copy()
    varia = stdevs > 0                  # máscara sobre as colunas
    rescaled[:, varia] = (X[:, varia] - means[varia]) / stdevs[varia]
    return rescaled

rescaled_xs = rescale(xs)
rescaled_xs[0].round(4)

A coluna de 1 sobrevive intacta, e agora dá para ver por quê: o desvio padrão dela é zero, `varia` não a marca, e a atribuição `rescaled[:, varia] = ...` simplesmente não a alcança. Nenhuma divisão por zero acontece porque a coluna nunca entra na conta.

> **❗ Importante — Reescalonar aqui é higiene; na próxima seção vira sobrevivência**
>
> Neste ajuste linear, esquecer o `rescale` produziria um modelo ruim — coeficientes que não convergem, uma taxa de aprendizado que serve para uma coluna e não para a outra. Chato, mas visível: o número sai errado e você percebe.
>
> Na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html), com o mesmo conjunto de dados e o mesmo tipo de otimizador, esquecer este passo não produz um modelo ruim. Produz uma perda igual a `nan` já no chute inicial — e, pior, um treino que roda até o fim mesmo assim e devolve coeficientes com cara de resposta. Vale voltar a esta linha depois de ver aquilo acontecer.

Agora o ajuste, com o `least_squares_fit` do [capítulo anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html), sem uma linha de mudança — só o gerador, que ele exige como argumento para que a semente fique visível aqui, ao lado do sorteio que ela produz:

In [ ]:
from scratch_np.multiple_regression import least_squares_fit, predict

rng = np.random.default_rng(0)
learning_rate = 0.001
beta = least_squares_fit(rescaled_xs, ys, rng, learning_rate, 1000, 1)

beta.round(4)

O modelo rodou e devolveu coeficientes. Vamos ver as previsões contra os valores reais. `predict(rescaled_xs, beta)` é `X @ beta` — as 200 previsões numa multiplicação de matriz por vetor, não uma por vez, do mesmo jeito que a [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) mostrou a mesma função aceitar um usuário ou a matriz inteira:

In [ ]:
# Figura: Regressão linear para prever contas pagas
previsoes = predict(rescaled_xs, beta)

plt.scatter(previsoes, ys, marker='.')
plt.axvline(0, color='gray', linestyle=':')
plt.axvline(1, color='gray', linestyle=':')
plt.xlabel("previsto")
plt.ylabel("real")
plt.title("Regressão linear para prever contas pagas")
plt.show()

### Os dois problemas

O primeiro salta do gráfico. As duas linhas pontilhadas marcam 0 e 1, os únicos valores que a variável a prever assume. Trinta e oito dos 200 pontos caem fora desse intervalo — e para contá-los basta uma máscara: `previsoes < 0` devolve um array de 200 booleanos, e `np.sum` conta os `True`.

In [ ]:
negativas = np.sum(previsoes < 0)
acima_de_um = np.sum(previsoes > 1)

print(f"previsões negativas:  {negativas}")
print(f"previsões acima de 1: {acima_de_um}")
print(f"mínimo: {previsoes.min():.4f}   máximo: {previsoes.max():.4f}")

Gostaríamos que a saída ficasse entre 0 e 1, para poder lê-la como probabilidade — uma previsão de 0,25 significaria "25% de chance de ser assinante". Isso seria perfeitamente aceitável. Mas as previsões deste modelo vão de **−0,58** a **1,44**: 37 dos 200 usuários recebem uma previsão negativa, e um recebe previsão acima de 1. Não há leitura possível para uma probabilidade negativa.

O segundo problema é mais sutil, e é sobre a validade do ajuste, não sobre a interpretação da saída. A regressão linear supõe que os erros são **não correlacionados** com as colunas de `x` — é uma das hipóteses que a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) listou. Aqui essa hipótese é violada por construção. O coeficiente de experiência é positivo (mais experiência, maior previsão), então o modelo produz valores altos para quem tem muita experiência. Só que o valor real é, no máximo, 1. Logo, uma previsão muito alta **obriga** o erro a ser muito negativo — o erro passa a depender de `x`, exatamente o que a hipótese proibia. A consequência não é cosmética: a estimativa de $\beta$ fica **enviesada**.

> **🔷 Conceito**
>
> O que queremos, no lugar disso: que valores grandes e positivos de `x_i @ beta` correspondam a probabilidades **próximas de 1**, e valores grandes e negativos, a probabilidades **próximas de 0** — com uma transição suave no meio.
>
> Repare no que essa formulação **não** pede. Ela não pede um modelo diferente, nem um otimizador diferente, nem abandonar o produto escalar. `x_i @ beta` continua sendo o coração da coisa; a estrutura linear do capítulo anterior sobrevive inteira. O que falta é uma função aplicada **depois** dele, que pegue a reta inteira dos reais e a comprima dentro de $[0, 1]$. É essa função que a [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) apresenta.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O ajuste que acabamos de fazer tem nome na literatura estatística — **modelo de probabilidade linear** — e não é uma bobagem completa: quando as probabilidades envolvidas ficam longe de 0 e de 1, ele aproxima razoavelmente bem e tem a vantagem de coeficientes diretamente interpretáveis. Fora dessa faixa, ele é o que você viu no gráfico.
>
> O ponto para esta seção é outro: **a biblioteca não avisa**. `LinearRegression().fit(X, y)` com `y` binário roda, converge, devolve `coef_` e `score`, e nenhuma linha de aviso menciona que o alvo só tem dois valores:
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> X = xs[:, 1:]                # sem a coluna de 1
> modelo = LinearRegression().fit(X, ys)
> modelo.predict(X).min()      # negativo
> ```
>
> A escolha certa para um alvo binário é `LogisticRegression` — que é o que este capítulo constrói do zero, e que aparece no callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html). Escolher entre as duas é decisão de quem modela, não da biblioteca.

## A Função Logística

> **📌 Nota**
>
> Esta seção corresponde a *The Logistic Function*, do capítulo 16 de Grus (2019).

A seção anterior terminou com um pedido preciso: uma função que receba a reta inteira dos reais e devolva um número em $[0, 1]$, crescente e suave. A escolha da regressão logística é a **função logística**:

In [ ]:
import numpy as np

def logistic(x):
    return 1.0 / (1 + np.exp(-x))

Uma linha — e ela vale tanto para um número quanto para um array inteiro, porque `np.exp` e a divisão trabalham elemento a elemento. É a mesma função, avaliada em um ponto ou em mil. É útil ver o formato dela antes de qualquer conta:

In [ ]:
# Figura: A função logística
from matplotlib import pyplot as plt

grade = np.linspace(-10, 10, 201)

plt.plot(grade, logistic(grade))
plt.axhline(0, color='gray', linewidth=0.8)
plt.axhline(1, color='gray', linewidth=0.8)
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
plt.axvline(0, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel("x")
plt.ylabel("logistic(x)")
plt.title("A função logística")
plt.show()

`logistic(grade)` é uma chamada só para os 201 valores da grade — não há laço nenhum entre a função e a curva.

Conforme a entrada fica grande e positiva, a saída chega cada vez mais perto de 1; conforme fica grande e negativa, chega cada vez mais perto de 0. Em zero ela vale exatamente 0,5. É exatamente a forma pedida na seção anterior.

Ela tem também uma propriedade conveniente: a derivada se escreve em função da própria função.

In [ ]:
def logistic_prime(x):
    y = logistic(x)
    return y * (1 - y)

Isso vai importar daqui a pouco — é o que torna o gradiente deste modelo simples o bastante para caber em uma linha.

Com ela, o modelo fica:

$$
y_i = f(\mathbf{x}_i \cdot \beta) + \varepsilon_i
$$

onde $f$ é a função logística. Repare no que **não** mudou: $\mathbf{x}_i \cdot \beta$ continua ali, com a coluna de 1 e tudo. A estrutura linear do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) sobreviveu inteira; ganhou uma função por cima.

### Por que minimizar quadrados deixa de ser a mesma coisa

Na regressão linear, ajustamos o modelo minimizando a soma dos erros ao quadrado, e a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) mostrou que isso acabava escolhendo o $\beta$ que **maximiza a verossimilhança dos dados** — não por coincidência, mas como consequência de supor erros normais.

Aqui as duas coisas deixam de ser equivalentes. O alvo é binário: um usuário paga ou não paga, e o "erro" de um ponto não é um desvio normalmente distribuído em torno de uma reta — é a diferença entre um rótulo em $\{0, 1\}$ e uma probabilidade em $[0, 1]$. Supor erros normais aqui seria supor algo que os dados desmentem por construção. Some a suposição, some a equivalência, e sobra apenas um dos dois lados: **a verossimilhança**. Vamos maximizá-la diretamente, por gradiente descendente.

Dado um $\beta$, o modelo diz que cada $y_i$ vale 1 com probabilidade $f(\mathbf{x}_i \cdot \beta)$ e 0 com probabilidade $1 - f(\mathbf{x}_i \cdot \beta)$. Essas duas frases se juntam numa só expressão:

$$
p(y_i \mid \mathbf{x}_i, \beta) = f(\mathbf{x}_i \cdot \beta)^{y_i} \, \bigl(1 - f(\mathbf{x}_i \cdot \beta)\bigr)^{1 - y_i}
$$

O truque é o expoente. Se $y_i$ é 0, o primeiro fator vira $f^0 = 1$ e sobra $1 - f(\mathbf{x}_i \cdot \beta)$; se $y_i$ é 1, o segundo fator vira 1 e sobra $f(\mathbf{x}_i \cdot \beta)$. Uma fórmula, os dois casos, sem `if`.

Essa expressão tem nome: é a distribuição de **Bernoulli** de parâmetro $f(\mathbf{x}_i \cdot \beta)$ — a distribuição de uma única moeda viciada. É exatamente a troca que a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) anunciou ao dizer que a máxima verossimilhança sobreviveria à mudança de alvo, mas a distribuição suposta para os dados não: lá era normal, aqui é Bernoulli. O método é o mesmo; a densidade que entra nele é outra, e é dela que vem tudo o que muda daqui para a frente.

Como na [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), é mais simples trabalhar com o **logaritmo** da verossimilhança:

$$
\log L(\beta \mid \mathbf{x}_i, y_i) = y_i \log f(\mathbf{x}_i \cdot \beta) + (1 - y_i) \log \bigl(1 - f(\mathbf{x}_i \cdot \beta)\bigr)
$$

Como o log é estritamente crescente, qualquer $\beta$ que maximize a log-verossimilhança maximiza também a verossimilhança, e vice-versa.

> **🔷 Conceito**
>
> Falta um último ajuste, e ele é de sinal. **O gradiente descendente minimiza**; a máquina construída no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) sabe descer, não subir. E maximizar uma função é exatamente o mesmo que minimizar o negativo dela.
>
> Por isso o que vamos otimizar se chama **log-verossimilhança negativa**. Não é uma quantidade nova: é a log-verossimilhança da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) com o sinal trocado, para que o otimizador que já temos possa ser usado sem modificação. Fora deste livro você vai encontrá-la com outro nome — *log loss*, ou *binary cross-entropy* —, que é a mesma função sob rótulos diferentes.

### A perda e o seu gradiente

Para um único ponto:

In [ ]:
def _negative_log_likelihood(x: np.ndarray, y: float, beta: np.ndarray) -> float:
    """A log-verossimilhança negativa de um único ponto"""
    if y == 1:
        return float(-np.log(logistic(x @ beta)))
    else:
        return float(-np.log(1 - logistic(x @ beta)))

Supondo que os pontos são independentes entre si, a verossimilhança do conjunto é o **produto** das verossimilhanças individuais — e portanto a log-verossimilhança é a **soma** dos logs:

In [ ]:
def negative_log_likelihood(X: np.ndarray, y: np.ndarray, beta: np.ndarray) -> float:
    p = logistic(X @ beta)
    return float(-np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

Repare no que a versão em array faz de diferente da de um ponto só. `X @ beta` calcula os 200 produtos escalares de uma vez, `logistic` os converte nas 200 probabilidades, e a expressão `y * np.log(p) + (1 - y) * np.log(1 - p)` é a fórmula do expoente escrita literalmente: quando `y` é 0, o primeiro termo é multiplicado por zero; quando é 1, o segundo é. O `if` do ponto isolado virou uma multiplicação por 0 ou 1 — e **guarde essa troca**, porque a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) mostra o preço dela. `np.sum` fecha a soma sobre os pontos, e o `float(...)` devolve um número Python em vez de um `np.float64`.

Um pouco de cálculo dá o gradiente — e ele cabe numa linha, com uma forma que você já viu:

In [ ]:
def negative_log_gradient(X: np.ndarray, y: np.ndarray, beta: np.ndarray) -> np.ndarray:
    return X.T @ (logistic(X @ beta) - y)

É a segunda vez que `X.T @ (alguma coisa por ponto)` aparece neste livro. Na [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) o vetor entre parênteses era o resíduo `X @ beta - y`; aqui é `logistic(X @ beta) - y`, o mesmo resíduo com a previsão passada pela logística. Transpor põe cada ponto numa coluna, multiplicar pesa a coluna $i$ pelo resíduo $i$, e o produto soma todas: `(d, n)` vezes `(n,)` dá `(d,)`, um número por coeficiente. As três funções que o Grus escreve — a parcial de um ponto, o gradiente de um ponto, a soma sobre os pontos — são as três etapas dessa única linha.

> **🟩 Exemplo**
>
> "Um pouco de cálculo" é uma frase que costuma esconder trabalho. Aqui não esconde muito, e vale fazer a conta — é ela que explica por que `logistic_prime` foi definida lá em cima.
>
> Escreva $f$ para $f(\mathbf{x} \cdot \beta)$. A perda de um ponto é
>
> $$
> -\log L = -y \log f - (1 - y)\log(1 - f)
> $$
>
> Derivando em relação a $\beta_j$: a derivada de $\mathbf{x} \cdot \beta$ em relação a $\beta_j$ é $x_j$ (o mesmo argumento da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html)), e a derivada de $f$ é $f(1-f)$ — a propriedade conveniente. Pela regra da cadeia:
>
> $$
> \frac{\partial}{\partial \beta_j}(-\log L) = -y\,\frac{f(1-f)}{f}\,x_j + (1-y)\,\frac{f(1-f)}{1-f}\,x_j
> $$
>
> Os cancelamentos são o ponto: o $f$ do denominador do primeiro termo mata o $f$ do numerador, e o $(1-f)$ do segundo faz o mesmo. Sobra
>
> $$
> \bigl[-y(1-f) + (1-y)f\bigr] x_j = (f - y)\, x_j
> $$
>
> que é exatamente `(p - y) * x_j`, com `p = logistic(x @ beta)` — ou seja, a coluna $j$ de `X.T @ (logistic(X @ beta) - y)`, ponto a ponto.
>
> Agora compare com o gradiente da regressão múltipla, `2 * batch_xs.T @ erros`, onde `erros` era `previsão - y`. **É a mesma forma:** resíduo vezes $x_j$. O que mudou foi o significado de "previsão" — antes um número qualquer, agora uma probabilidade — e o fator 2 desapareceu no caminho. Se a estrutura do gradiente parece familiar, é porque ela é.

### Uma armadilha que ainda vai explodir

Olhe de novo o `negative_log_likelihood`. Ele calcula `np.log(1 - p)` para **todos** os pontos, inclusive aqueles em que `y` vale 1 e o termo será multiplicado por zero. Isso pressupõe que `logistic` nunca devolva exatamente `1.0` — porque `log(0)` não existe.

Matematicamente, a pressuposição está certa: a função logística é estritamente menor que 1 para todo $x$ finito. Em `float64`, está errada:

In [ ]:
for x in [10, 20, 30, 35, 36, 37, 40]:
    p = float(logistic(x))
    print(f"logistic({x:>2}) = {p!r:<24} 1 - logistic({x:>2}) = {1 - p!r}")

> **⚠️ Atenção — `logistic(37)` é `1.0`, e isso não é arredondamento de exibição**
>
> A partir de $x \approx 36{,}74$, `logistic(x)` devolve **exatamente** `1.0` — o `repr` acima mostra que não há dígitos escondidos. A razão é aritmética de ponto flutuante: `np.exp(-37)` vale cerca de $8{,}5 \times 10^{-17}$, e somar isso a 1 em `float64` não muda nada. O menor incremento representável acima de 1 é $2^{-52} \approx 2{,}2 \times 10^{-16}$, e uma parcela que não passa da **metade** disso ($1{,}11 \times 10^{-16}$) desaparece no arredondamento. O denominador vira `1.0` exato, e `1/1.0` é `1.0`.
>
> Dá para escrever o limiar exato. A saturação acontece quando `np.exp(-x)` fica menor ou igual a $2^{-53}$, e portanto quando
>
> $$
> x \;\geq\; -\log\bigl(2^{-53}\bigr) \;=\; 53 \ln 2 \;\approx\; 36{,}7368
> $$
>
> que é o 36,74 medido acima, agora até o último dígito — conferido por bissecção sobre a própria `logistic`. Não é um valor mágico da função logística; é a mantissa do `float64` aparecendo, com o sinal trocado por causa do `-x` no expoente.
>
> Consequência direta: `1 - logistic(37)` é `0.0`. E aqui o `numpy` se comporta de um jeito que vale conhecer antes de vê-lo acontecer: `np.log(0.0)` **não** levanta exceção. Ela devolve `-inf` e emite um `RuntimeWarning: divide by zero encountered in log` — uma linha no stderr, que some no meio de uma saída longa e não interrompe coisa nenhuma. Pior: `-inf` multiplicado pelo zero do outro termo dá `nan`, com outro aviso, e aí a soma inteira vira `nan`.
>
> Isso não é uma curiosidade sobre ponto flutuante. É uma bomba armada no código que acabamos de escrever — e o incômodo é que ela não explode: ela vaza, em silêncio, na próxima seção.
>
> O outro extremo tem um limite parecido. Para $x$ abaixo de cerca de −709,78 — o logaritmo do maior `float64` —, `np.exp(-x)` estoura o alcance do tipo e devolve `inf`, com um `RuntimeWarning: overflow encountered in exp`. Como `1 / (1 + inf)` é `0.0`, `logistic(-800)` devolve zero, que é o valor certo; o que se perde não é o número, é a garantia de que nada estranho aconteceu. Numa biblioteca em C isso seria uma exceção; em `numpy` é um aviso, e aviso é coisa que se ignora.

> **💡 Dica — Na prática: `scipy` e `scikit-learn`**
>
> Nenhuma biblioteca séria calcula `log(1 - sigmoid(x))` do jeito que acabamos de escrever, e vale ver as duas saídas que elas usam.
>
> A primeira é ter uma versão da logística que não estoura. O `scipy` traz `scipy.special.expit`, que é a mesma função com um nome antigo (*expit* é o inverso do *logit*), implementada em C:
>
> ```python
> from scipy.special import expit
>
> expit(800)     # 1.0
> expit(-800)    # 0.0, sem aviso nenhum
> ```
>
> Repare que `expit(-800)` devolve `0.0` **sem aviso nenhum** — o nosso também devolve `0.0`, mas depois de estourar o expoente e reclamar. A diferença é de ruído, não de resultado, e nem essa é a diferença que importa: trocar `logistic` por `expit` **não** resolveria o problema desta seção, porque `expit(37)` também é `1.0` exato e `log(1 - 1.0)` continua sendo `log(0)`.
>
> A segunda saída é a que realmente resolve, e é conceitual: **nunca calcular a probabilidade para depois tirar o log**. O `scipy` tem `log_expit`, que devolve $\log f(x)$ direto, sem passar pelo número intermediário:
>
> ```python
> from scipy.special import log_expit
>
> log_expit(-800)    # -800.0, exato
> ```
>
> `np.log(expit(-800))` devolveria `-inf`; `log_expit(-800)` devolve `-800.0` sem esforço. É a mesma ideia da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), onde a log-verossimilhança foi calculada como soma de logs em vez de log de um produto: o número minúsculo nunca chega a existir na memória.
>
> Há ainda uma terceira postura, mais bruta, e é a que a métrica pronta do `scikit-learn` adota. `sklearn.metrics.log_loss` **corta** as probabilidades para dentro de $[\epsilon, 1-\epsilon]$ antes de tirar o log, com $\epsilon$ igual ao épsilon de máquina. Ela nunca levanta exceção: dar a ela uma probabilidade de `1.0` para um rótulo `0` devolve cerca de 36,04, que é $-\log(2{,}2 \times 10^{-16})$. É um número grande, sinalizando um erro grave, mas é um número — e é isso que ela devolve em vez do `nan` que o nosso código vai produzir na próxima seção, e `nan`, ao contrário de 36,04, contamina qualquer conta que o toque.
>
> As três abordagens são defensáveis. A que **não** é defensável é a nossa, e ela está aqui de propósito: o que você vai ver na próxima seção — uma perda que não é um número, e um treino que roda até o fim assim mesmo — é o que essas bibliotecas gastam código para que você nunca encontre.

## Aplicando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Applying the Model*, do capítulo 16 de Grus (2019).

Temos todas as peças: os dados, a função logística, a perda e o seu gradiente. Falta ajustar — e é aqui que este capítulo se separa dos dois anteriores.

> **❗ Importante — Não existe fórmula fechada para esta**
>
> A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) ajustou uma reta com duas médias, uma covariância e uma variância — sem laço nenhum. A [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) usou gradiente descendente para a regressão múltipla, mas deixou registrado que **a solução exata existia**: bastaria resolver um sistema linear, o que aquele capítulo não fez por decisão de escopo, não por impossibilidade.
>
> Aqui a situação muda de natureza. A condição de otimalidade da regressão logística — o gradiente igualado a zero — envolve $\beta$ **dentro** de uma exponencial, e não há manipulação algébrica que isole $\beta$ de lá. Não é uma conta pesada demais para caber num chunk: é uma conta que não existe. A [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) prometeu que este dia chegaria, e chegou. Para este modelo — e para as redes neurais dos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) —, o gradiente descendente deixa de ser a alternativa prática e passa a ser o único caminho até um ajuste.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from scratch_np.logistic_regression import (
    xs, ys, logistic, _negative_log_likelihood,
    negative_log_likelihood, negative_log_gradient,
)

xs.shape, xs[0], ys[0]

São o mesmo `logistic`, o mesmo `negative_log_likelihood` e o mesmo `negative_log_gradient` que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) construiu, agora junto com os dados — e também o `_negative_log_likelihood`, a versão de um ponto só, que vai ser útil daqui a pouco.

### O que acontece se você simplesmente ajustar

O gradiente descendente precisa de um ponto de partida. Como sempre, um chute aleatório, com semente fixa:

In [ ]:
rng = np.random.default_rng(0)
beta = rng.random(3)

beta

Antes de gastar 5.000 passos, vale avaliar a perda uma vez, nesse ponto de partida. É a conta mais barata do capítulo: `negative_log_likelihood(xs, ys, beta)`. Capturamos os avisos de propósito, para imprimi-los junto com o resultado. Fora de um livro eles iriam para o stderr, misturados com o resto — e a razão de estarem aqui, em destaque, é que eles são a **única** coisa que o programa vai dizer sobre o que acabou de acontecer.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    perda = negative_log_likelihood(xs, ys, beta)

print(f"perda inicial: {perda}")
for aviso in avisos:
    print(f"   {aviso.category.__name__}: {aviso.message}")

Não houve ajuste ruim, convergência lenta nem coeficiente estranho. Também não houve exceção, mensagem de erro ou parada. **A perda não é um número** — é `nan`, o valor que o `float64` reserva para "isto não faz sentido" — e o programa seguiu em frente sem se importar. Vale rastrear exatamente onde.

In [ ]:
for i in [0, 1]:
    d = xs[i] @ beta
    print(f"ponto {i}: x = {xs[i]}, y = {ys[i]:.0f}")
    print(f"   x @ beta = {d:.2f}")
    print(f"   logistic(...) = {float(logistic(d))!r}")
    print(f"   1 - logistic(...) = {float(1 - logistic(d))!r}")

O culpado é o **salário**. O primeiro usuário tem 48.000, e o coeficiente inicial sorteado para essa coluna foi 0,041 — o produto escalar dá 1.967,55, dominado inteiramente por aquela parcela, que sozinha vale 1.966,73. A experiência, que vale 0,7, contribui com 0,19; o termo constante, com 0,64. As outras duas colunas simplesmente não participam da conta.

E `logistic(1967.55)` é `1.0`, exato, pela razão que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) mediu: qualquer entrada acima de 36,74 satura. Não faltou pouco: sobrou por um fator de mais de cinquenta.

Repare no que acontece com os dois primeiros pontos, porque eles falham de maneiras diferentes. Usamos aqui a versão de um ponto só, `_negative_log_likelihood`, porque ela tem o `if` que separa os dois casos. A versão em array não tem, e é essa a diferença:

In [ ]:
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    for i in [0, 1]:
        perda_i = _negative_log_likelihood(xs[i], ys[i], beta)
        print(f"ponto {i} (y = {ys[i]:.0f}): perda = {perda_i!r}")

for aviso in avisos:
    print(f"   {aviso.category.__name__}: {aviso.message}")

> **⚠️ Atenção — Três números, e nenhum deles interrompe o programa**
>
> O ponto 0 tem `y = 1`, então a perda dele é `-np.log(logistic(...))`, que é `-log(1.0)`, que é **`-0.0`**. Nenhum erro: o modelo declara certeza absoluta de que este usuário paga, e a perda registra zero — perfeição — para um $\beta$ que foi **sorteado ao acaso segundos atrás**.
>
> O ponto 1 tem `y = 0`. A perda dele é `-np.log(1 - logistic(...))`, e `1 - 1.0` é `0.0`. `np.log(0.0)` é `-inf`, então a perda deste ponto é **`inf`**. Um infinito é um mau resultado honesto: ele diz "este ponto é impossível sob o seu modelo", e diria a mesma coisa somado a qualquer outra perda finita.
>
> Já a perda do conjunto, calculada pela versão em array, é **`nan`** — e é o pior dos três. Ela não vem do ponto 1; vem do ponto **0**. A fórmula vetorizada calcula as duas parcelas para todos os pontos e depois zera a que não vale: no ponto 0, `(1 - y)` é 0 e `np.log(1 - p)` é `-inf`, e `0 * -inf` é `nan`. Foi o ponto **bem classificado** que envenenou a soma. E `nan` é contagioso: qualquer conta que o toque devolve `nan`, e qualquer comparação com ele é falsa — inclusive `nan > 1000`, inclusive `nan == nan`.
>
> Nenhum dos três interrompe nada. O código de Grus (2019), com `math.log`, receberia um `ValueError: math domain error` no ponto 1 e pararia ali: barulhento, feio e **impossível de ignorar**. Em `numpy` as contas desta página produzem três avisos e seguem. Trocar a exceção pelo aviso é o preço da vetorização, e é um preço que quase todo código científico paga sem perceber que pagou.

Com o `math.log` do Grus (2019) a história terminaria aqui, com uma exceção na tela. Em `numpy` ela continua — e é a continuação que interessa. O gradiente `X.T @ (logistic(X @ beta) - y)` **nunca** é `nan`: `logistic` devolve um número entre 0 e 1 mesmo saturada, e a subtração de `y` é finita. É só a *perda* que virou `nan`, e a perda não entra no cálculo do passo. Então o treino roda.

No livro, os avisos deste chunk estão escondidos, e vale dizer o que se esconde: os 5.000 passos emitem alguns **milhares** de avisos, de três tipos que você já conhece — o `divide by zero` e o `invalid value` da célula acima, e o `overflow encountered in exp` que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) previu. Mas essa supressão quase não muda o que apareceria na tela, porque o Python imprime cada aviso **uma vez por linha de código** e cala as repetições: sem esconder nada, o treino inteiro deixaria três linhas, no meio de qualquer outra saída.

In [ ]:
from scratch_np.gradient_descent import gradient_step

beta_cru = beta.copy()          # o mesmo chute inicial de cima
for _ in range(5000):
    beta_cru = gradient_step(beta_cru, negative_log_gradient(xs, ys, beta_cru), -0.01)

print(f"beta 'ajustado': {beta_cru.round(2)}")
print(f"perda final:     {negative_log_likelihood(xs, ys, beta_cru)}")

previsto = logistic(xs @ beta_cru) >= 0.5
print(f"acertos: {np.sum(previsto == (ys == 1))} de {len(ys)}")
print(f"quantos usuários o modelo classifica como pagantes: {np.sum(previsto)}")

Cinco mil passos, nenhuma exceção, um vetor de coeficientes no fim. E o modelo que ele descreve atribui probabilidade `0.0` a **todos** os 200 usuários: a saturação virou para o outro lado, e o classificador prevê "não paga" para a base inteira. Os 148 acertos são exatamente os 148 usuários que não pagam — é o mesmo desempenho de responder "não" sem olhar dado nenhum, que a [seção 8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) usou como piso.

Some a exceção, some o aviso na hora certa, e o que sobra é um vetor de três números com cara de resposta. Se você tivesse impresso só `beta_cru`, não haveria nada na tela indicando que o ajuste inteiro é lixo.

Vale parar aqui, porque este é o ponto do capítulo que sobrevive depois que o assunto "regressão logística" for esquecido.

`LogisticRegression().fit(X, y)` não teria produzido **nenhum** dos três números. Nem o `nan`, nem o `inf`: a biblioteca calcula a perda por rotinas numericamente estáveis, que nunca chegam a formar `1 - 1.0`. Nem o `-0.0`: ela sequer expõe a perda de um ponto isolado — devolve coeficientes, previsões e um `score`, e mais nada. Os números que você acabou de ver **só são visíveis de dentro**. De fora, o mesmo ajuste teria rodado até o fim, sem uma linha de aviso, e devolvido um vetor de coeficientes com cara de resposta.

Generalize, porque a lição não é sobre a função logística: **o número na sua tela pode ser mentira, e a tela não tem como avisar.** Uma perda de zero é o que você esperaria de um modelo perfeito e é também o que sai de um $\beta$ sorteado ao acaso quando a aritmética satura. Os dois casos produzem o mesmo `0.0`, indistinguíveis. Quem só chama a função não tem de onde tirar a diferença; quem sabe que a logística tem um teto em `float64` e que a perda vira `-log(1.0)` sabe onde olhar. É por isso que esta disciplina constrói o código do zero, e é o motivo de esta seção ser a mais importante do capítulo.

### O conserto

O conserto é o mesmo passo que a [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) já tinha dado por higiene, e que aqui é a diferença entre existir um modelo e existir um vetor de números que finge ser um: **reescalonar**. A função é a da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html), repetida porque esta página é um kernel próprio — o mesmo código da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html):

In [ ]:
from typing import Tuple

def scale(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """devolve a média e o desvio padrão de cada coluna"""
    X = np.asarray(X, dtype=float)
    return X.mean(axis=0), X.std(axis=0, ddof=1)

def rescale(X: np.ndarray) -> np.ndarray:
    """cada coluna passa a ter média 0 e desvio padrão 1"""
    X = np.asarray(X, dtype=float)
    means, stdevs = scale(X)

    rescaled = X.copy()
    varia = stdevs > 0                  # máscara sobre as colunas
    rescaled[:, varia] = (X[:, varia] - means[varia]) / stdevs[varia]
    return rescaled

rescaled_xs = rescale(xs)
rescaled_xs[0].round(4)

O mesmo primeiro ponto, o mesmo $\beta$ sorteado, agora sem drama:

In [ ]:
d = rescaled_xs[0] @ beta
print(f"x @ beta = {d:.4f}")
print(f"logistic(...) = {float(logistic(d)):.4f}")
print(f"perda no conjunto todo = {negative_log_likelihood(rescaled_xs, ys, beta):.4f}")

> **🔷 Conceito**
>
> De 1.967,55 para 0,1804. Nada mudou no modelo, no otimizador ou nos dados — a mesma pessoa, com a mesma experiência e o mesmo salário, continua ali. O que mudou foi a **unidade**: salário deixou de ser medido em reais e passou a ser medido em desvios padrão a partir da média.
>
> Vale enunciar por que a diferença é tão violenta aqui e era só incômoda na regressão linear. Um modelo linear não tem teto: se o produto escalar der 20.000, a previsão é 20.000 — um número absurdo, mas um número, e o gradiente ainda aponta para algum lugar. A logística tem teto, e o alcance útil dela é aproximadamente $[-37, 37]$. Fora dessa janela, a função é **constante** em `float64`: derivada zero, gradiente zero, nenhuma informação sobre para onde ir. Reescalonar não é uma otimização de desempenho; é o que coloca os dados dentro da faixa em que a função ainda tem inclinação.
>
> E a perda do conjunto virou um número: 176,21. Alta — são 200 pontos e um $\beta$ aleatório —, mas finita, comparável e capaz de descer.

### O ajuste

Separamos treino e teste com o `train_test_split` do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) e descemos o gradiente:

In [ ]:
from scratch_np.machine_learning import train_test_split

rng = np.random.default_rng(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33, rng)

len(x_train), len(x_test)

> **⚠️ Atenção — A semente não é opcional**
>
> `np.random.default_rng(0)` aparece pela segunda vez nesta página, e não é descuido. O gerador do começo da seção já tinha sorteado o chute que usamos para dissecar a saturação; se a divisão treino/teste saísse **dele**, ela dependeria de quantos diagnósticos rodamos antes — acrescentar uma célula de investigação mudaria o conjunto de teste, o que é a última coisa que se quer. Recriar o gerador com a mesma semente torna esta célula independente de tudo o que veio acima. É o que Grus (2019) faz, chamando `random.seed(0)` uma segunda vez no mesmo trecho.
>
> Daqui para baixo, porém, o gerador é **um só**: `train_test_split` consome dele a permutação que separa treino e teste, e o `beta` do ajuste é sorteado a seguir, do gerador já avançado. Por isso ele não é o `[0.637, 0.270, 0.041]` de cima — é o próximo sorteio da mesma sequência, `[0.6526, 0.2738, 0.7027]`.
>
> Duas consequências, e as duas importam nesta página. A semente decide **quem** vai para o teste, e com 66 pontos isso move a matriz de confusão da [seção 13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html) mais do que a maioria das decisões de modelagem que você vai tomar. E decide de onde o gradiente parte — o que aqui quase não muda o destino, porque a log-verossimilhança negativa da regressão logística é convexa e tem um mínimo só.
>
> A semente está escrita no chunk, ao lado do sorteio que ela produz, e não num estado global algumas células acima. É por isso que `train_test_split` exige o gerador como argumento em vez de criar um por conta própria.

134 pontos de treino, 66 de teste. Agora o laço, que é o gradiente descendente em lote completo do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) — sem minibatches, porque com 134 pontos e três parâmetros não há razão para complicar. O laço percorre os **passos**, que são o algoritmo; os 134 pontos não aparecem nele. Eles estão dentro do `X.T @ (...)` do gradiente, somados de uma vez:

In [ ]:
learning_rate = 0.01

# ponto de partida aleatório, sorteado depois da divisão
beta = rng.random(3)
perdas = []
betas = []          # guardamos o caminho inteiro; ele reaparece mais abaixo

for epoch in range(5000):
    gradient = negative_log_gradient(x_train, y_train, beta)
    beta = gradient_step(beta, gradient, -learning_rate)
    perdas.append(negative_log_likelihood(x_train, y_train, beta))
    betas.append(beta)

perdas = np.array(perdas)
betas = np.array(betas)

beta.round(4), round(perdas[-1], 4)

O $\beta$ ajustado é aproximadamente `[-2.35, 4.14, -4.07]`, com perda final de 39,78 no conjunto de treino. `betas` sai do laço como uma lista de 5.000 vetores e vira uma matriz `(5000, 3)` — uma linha por epoch, uma coluna por coeficiente. Vale ver como a perda chegou lá:

In [ ]:
# Figura: Log-verossimilhança negativa no conjunto de treino, por epoch (escala logarítmica no eixo x)
plt.plot(np.arange(1, len(perdas) + 1), perdas)
plt.xscale('log')
plt.xlabel("epoch (escala logarítmica)")
plt.ylabel("log-verossimilhança negativa")
plt.title("Convergência do ajuste")
plt.show()

In [ ]:
for e in [1, 10, 100, 1000, 5000]:
    print(f"epoch {e:>5}: perda = {perdas[e - 1]:.4f}")

A queda é quase toda no começo. Depois do primeiro epoch a perda está em 102,70; no décimo já caiu para 58,35; no centésimo, 40,77 — a 2,5% do valor final. No milésimo o número já é 39,7766, e os 4.000 epochs restantes movem a perda da sétima casa decimal para baixo: de 39,77661775 para 39,77661701, uma diferença de $7{,}4 \times 10^{-7}$. Quem ainda se mexe na terceira casa é o $\beta$, de 4,1401 para 4,1411 na segunda coordenada. É o padrão que a [seção 5.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) descreveu — queda rápida no início, cada vez mais lenta perto do mínimo — e é por isso que 5.000 passos aqui são generosidade, não necessidade.

### Voltando às unidades originais

Os coeficientes acima valem para os dados reescalonados. Dá para convertê-los de volta às unidades do mundo, desfazendo a transformação:

In [ ]:
means, stdevs = scale(xs)

beta_unscaled = np.array([beta[0]
                          - beta[1] * means[1] / stdevs[1]
                          - beta[2] * means[2] / stdevs[2],
                          beta[1] / stdevs[1],
                          beta[2] / stdevs[2]])

beta_unscaled

E dá para verificar que os dois vetores descrevem o **mesmo** modelo, comparando a verossimilhança que cada um atribui ao conjunto completo. Grus (2019) faz isso com um `assert` de igualdade exata entre dois `float`. **Aqui ele não passa** — e por isso o nosso pede tolerância:

In [ ]:
esquerda = negative_log_likelihood(xs, ys, beta_unscaled)
direita = negative_log_likelihood(rescaled_xs, ys, beta)

assert np.isclose(esquerda, direita)

esquerda, direita, esquerda == direita

Os dois números diferem no último dígito, e `==` devolve `False`. Antes de concluir qualquer coisa disso, vale rodar a mesma comparação com outros $\beta$ — os do meio do treino, que ficaram guardados em `betas`:

In [ ]:
def desescalonar(b: np.ndarray) -> np.ndarray:
    """a mesma conversão de cima, para um beta qualquer"""
    return np.array([b[0] - b[1] * means[1] / stdevs[1] - b[2] * means[2] / stdevs[2],
                     b[1] / stdevs[1],
                     b[2] / stdevs[2]])

def as_duas_perdas(b: np.ndarray) -> Tuple[float, float]:
    """a perda no conjunto completo, calculada nas duas escalas"""
    return (negative_log_likelihood(xs, ys, desescalonar(b)),
            negative_log_likelihood(rescaled_xs, ys, b))

for e in [100, 500, 1000, 2000, 3000, 4000, 5000]:
    esquerda, direita = as_duas_perdas(betas[e - 1])
    print(f"epoch {e:>5}: {esquerda!r:<19} {direita!r:<19} "
          f"iguais? {esquerda == direita}")

pares = np.array([as_duas_perdas(b) for b in betas])
print(f"\nnos 5.000 epochs deste treino, comparando as duas escalas:")
print(f"   ==          vale em {np.sum(pares[:, 0] == pares[:, 1])}")
print(f"   np.isclose  vale em {np.sum(np.isclose(pares[:, 0], pares[:, 1]))}")

> **⚠️ Atenção — O `assert` do livro-texto não passa aqui — e não é defeito nosso**
>
> Aquele `assert` compara dois `float` por **igualdade exata**. No Grus (2019) ele passa; no nosso ajuste, com outro sorteio e outra ordem de operações, ele falha por uma unidade na última casa: 58,827241891663306 de um lado, 58,8272418916633 do outro. Sete dos sete epochs da tabela acima falham igual, e no treino inteiro a igualdade vale em **921 dos 5.000** — menos de um em cinco. **É cara ou coroa**, e o Grus tirou cara.
>
> O motivo é que as duas contas **não** se cancelam ponto a ponto. Tome o $\beta$ final e compare `xs @ beta_unscaled` com o produto escalar na escala reescalonada, usuário por usuário: só **33** dos 200 pares saem bit a bit idênticos. Os outros 167 diferem na última casa — nada maior que $5{,}3 \times 10^{-15}$, mas diferem. A conversão é linear no papel; em `float64` ela é uma sequência **diferente** de multiplicações e subtrações, e duas sequências com o mesmo valor exato não têm por que produzir o mesmo valor arredondado.
>
> A soma final às vezes salva a aparência e às vezes não: somados 200 termos com erros de última casa, o total cai no mesmo `float64` em 921 dos 5.000 casos e no vizinho nos outros 4.079. Nenhum dos 5.000 tem nada de errado com o modelo, com a conversão ou com os dados.
>
> É o mesmo construto que a [seção 10.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/04-testando-o-modelo.html) já tinha exibido. Lá o não determinismo vinha da ordem de iteração de um `set`; aqui vem da ordem das operações aritméticas. É a mesma lição com outra fachada: **`==` entre `float` é a pergunta errada.** A pergunta certa tem tolerância, e a linha de cima mostra o placar dela — `np.isclose` vale nos 5.000 epochs, sem exceção.
>
> Fica também um aviso sobre `assert` em geral, e ele ficou mais afiado do que estava. Aquele `assert` do livro-texto passa **sempre** que a semente é a mesma, porque aí o $\beta$ final também é. Isso não faz dele um teste: um `assert` que passa por construção e um `assert` que passa porque o sorteio caiu bem são coisas diferentes, e só a diferença de semente entre este livro e o do Grus foi suficiente para separar as duas.

### Interpretando os coeficientes

Infelizmente, estes coeficientes não são tão fáceis de ler quanto os de uma regressão linear. Mantendo tudo o mais constante:

- um ano a mais de experiência **adiciona 1,45 à entrada** da função logística;
- dez mil a mais de salário **subtrai 2,62 da entrada** da função logística.

O impacto disso na *saída*, porém, depende de onde a entrada já estava. Se `x_i @ beta` já é grande — probabilidade perto de 1 —, aumentá-lo ainda mais quase não muda nada; se está perto de zero, um empurrão pequeno muda bastante. É a curva em S da seção anterior aparecendo na interpretação: **o mesmo aumento no argumento vale coisas diferentes em pontos diferentes**.

In [ ]:
for exp, sal in [(2.0, 60000), (5.0, 57800), (8.0, 60000), (8.0, 100000)]:
    p = logistic(np.array([1.0, exp, sal]) @ beta_unscaled)
    print(f"{exp:4.1f} anos de experiência, salário {sal:>6} -> P(paga) = {p:.4f}")

O que dá para afirmar com segurança é o sinal: mantendo tudo o mais constante, **mais experiência aumenta** a chance de pagar, e **salário maior diminui**. O segundo sinal parece estranho até você lembrar do gráfico da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html), em que os pagantes se concentravam embaixo à direita. Com 8 anos de experiência, um salário de 60.000 dá 97,76% de chance de conta paga; o mesmo profissional ganhando 100.000 cai para 0,12%.

> **⚠️ Atenção**
>
> Nada disso é uma afirmação causal. O modelo não diz que aumentar o salário de alguém faria a pessoa cancelar a assinatura. Ele diz que, **nestes 200 usuários**, quem ganha mais tende a não assinar — e a explicação plausível (quem tem experiência e ganha pouco está procurando emprego) é uma história que nós contamos, não algo que o ajuste tenha demonstrado. É a mesma advertência da [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html), e ela não fica mais fraca porque a saída agora é uma probabilidade.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import LogisticRegression
>
> X = xs[:, 1:]                    # sem a coluna de 1
> modelo = LogisticRegression().fit(X, ys)
>
> modelo.intercept_, modelo.coef_
> ```
>
> Três coisas que vale saber, agora que você viu o código por dentro.
>
> **A biblioteca não quebra com os dados brutos — e também não teria mostrado a falha silenciosa.** Rodar o código acima nos `xs` sem reescalonar funciona: ela devolve intercepto 8,41 e coeficientes 1,51 e −0,00027, na mesma vizinhança do nosso `beta_unscaled` — `[7.87, 1.45, -0.000262]`. Isso não contradiz nada do que esta seção mostrou; é o resultado de um trabalho de engenharia que a implementação faz e a nossa não. O otimizador padrão (`lbfgs`) usa curvatura de segunda ordem, e por isso não sofre com colunas de escalas diferentes do jeito que um passo fixo sofre; e a perda é calculada por rotinas numericamente estáveis, que nunca formam `1 - 1.0`. A saturação em `float64` continua existindo — ela é uma propriedade do tipo, não do código —, mas nenhuma linha do seu programa a encontra.
>
> Repare que isso vale para os **três** números. O `nan` e o `inf` somem porque a perda é calculada de outro jeito, por rotinas que nunca formam `1 - 1.0`; o `-0.0` some porque a interface não tem onde exibi-lo — `LogisticRegression` devolve coeficientes, previsões e um `score`, nunca a perda de um ponto isolado. Nenhum dos três existe do lado de fora, e nem os avisos existem: com o otimizador da biblioteca o treino sem reescalonar converge, em vez de sair passeando até `-29999`. Os três números são, literalmente, o que você paga para ver o mecanismo.
>
> **Ela regulariza por padrão, e você não pediu.** `LogisticRegression` aplica penalidade L2 com `C=1.0` a menos que você diga o contrário — a mesma regularização *ridge* da [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html), aqui ligada de fábrica. É por isso que os coeficientes dela saem um pouco encolhidos em relação aos nossos. Aumentar `C` afrouxa a penalidade; com ela praticamente desligada (`C=1e6`) e os mesmos 134 pontos de treino reescalonados, a biblioteca devolve `[-2.34845, 4.14020, -4.07182]` contra os nossos `[-2.34854, 4.14109, -4.07339]`: iguais até a segunda casa decimal. Aqui os 5.000 passos de tamanho fixo chegaram onde o otimizador de segunda ordem chega, e não é coincidência — a log-verossimilhança negativa é convexa, tem um mínimo só, e nada além de paciência separa os dois métodos dele.
>
> **Reescalonar continua sendo boa ideia mesmo assim** — não pelo `nan`, que a biblioteca não produz, mas pela penalidade. Uma penalidade que soma quadrados de coeficientes trata todas as colunas com o mesmo rigor, e um coeficiente medido em "por real de salário" é numericamente minúsculo perto de um medido em "por ano de experiência". Sem reescalonar, a regularização pune as duas colunas de forma desigual por um motivo que não tem nada a ver com o problema. O `StandardScaler` da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) é a resposta.

## Qualidade do Ajuste

> **📌 Nota**
>
> Esta seção corresponde a *Goodness of Fit*, do capítulo 16 de Grus (2019).

Ainda não usamos os 66 pontos de teste que a seção anterior separou. Chegou a hora — e note que a pergunta desta seção é diferente da do capítulo passado. Lá, a qualidade do ajuste era o R², a fração da variação de `y` que o modelo capturava. Aqui `y` não tem variação a ser explicada: ele vale 0 ou 1. O que se mede é **quantas vezes o modelo acerta, e de que jeito ele erra** — o vocabulário do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html).

O `beta` que o resto desta seção usa é o ajuste da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) — o mesmo reescalonamento, a mesma divisão treino/teste, os mesmos 5.000 epochs com taxa 0,01. O código está à vista para deixar claro que é o mesmo ajuste, e não um parecido: mesma semente, mesma ordem de sorteios, mesmo `beta` até a última casa. Cada página deste livro roda num kernel próprio, então nada atravessa de uma seção para a outra sem ser reescrito.

In [ ]:
import numpy as np
from typing import Tuple
from matplotlib import pyplot as plt

from scratch_np.logistic_regression import (
    xs, ys, logistic, negative_log_gradient,
)
from scratch_np.gradient_descent import gradient_step
from scratch_np.machine_learning import train_test_split

def scale(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """devolve a média e o desvio padrão de cada coluna"""
    X = np.asarray(X, dtype=float)
    return X.mean(axis=0), X.std(axis=0, ddof=1)

def rescale(X: np.ndarray) -> np.ndarray:
    """cada coluna passa a ter média 0 e desvio padrão 1"""
    X = np.asarray(X, dtype=float)
    means, stdevs = scale(X)

    rescaled = X.copy()
    varia = stdevs > 0                  # máscara sobre as colunas
    rescaled[:, varia] = (X[:, varia] - means[varia]) / stdevs[varia]
    return rescaled

rescaled_xs = rescale(xs)

rng = np.random.default_rng(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33, rng)

beta = rng.random(3)
for _ in range(5000):
    beta = gradient_step(beta, negative_log_gradient(x_train, y_train, beta), -0.01)

beta.round(4)

### As quatro caixas

O modelo devolve uma **probabilidade**, não um rótulo. Para contar acertos é preciso primeiro transformá-la numa decisão, e a regra mais simples é a mais óbvia: prever "conta paga" sempre que a probabilidade passar de 0,5. Duas máscaras — o que o modelo previu e o que era verdade — e as quatro caixas são as quatro combinações de `&` e `~` entre elas, contadas de uma vez, como a [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html) montou a matriz de confusão do filtro de spam. O `if/elif/elif/else` do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) vira quatro linhas independentes, e cada uma diz na cara o que conta.

In [ ]:
previsoes = logistic(x_test @ beta)
previsto = previsoes >= 0.5          # a decisão: probabilidade acima de 0,5
real = y_test == 1

true_positives = np.sum(previsto & real)    # VP: paga e previmos que paga
false_positives = np.sum(previsto & ~real)  # FP: não paga e previmos que paga
false_negatives = np.sum(~previsto & real)  # FN: paga e previmos que não paga
true_negatives = np.sum(~previsto & ~real)  # VN: não paga e previmos que não paga

true_positives, false_positives, false_negatives, true_negatives

Quinze verdadeiros positivos, **nenhum** falso positivo, sete falsos negativos e 44 verdadeiros negativos. Em forma de tabela:

|  | Paga | Não paga |
|---|---|---|
| **Previu "paga"** | 15 | 0 |
| **Previu "não paga"** | 7 | 44 |

A coluna do zero merece uma pausa. Neste conjunto de teste o modelo nunca acusou de pagante quem não paga — todo erro dele é do outro tipo. Isso não é virtude do método: é o sorteio da divisão, e a [seção 13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) mostra que sobre os 200 usuários inteiros os dois tipos de erro aparecem. Com 66 pontos de teste, uma métrica é um número com muito menos casas confiáveis do que ele exibe.

E as métricas do Capítulo 8, importadas de onde foram escritas:

In [ ]:
from scratch_np.machine_learning import accuracy, precision, recall, f1_score

tp, fp = true_positives, false_positives
fn, tn = false_negatives, true_negatives

print(f"acurácia:  {accuracy(tp, fp, fn, tn):.4f}")
print(f"precisão:  {precision(tp, fp, fn, tn):.4f}")
print(f"revocação: {recall(tp, fp, fn, tn):.4f}")
print(f"F1:        {f1_score(tp, fp, fn, tn):.4f}")

**Precisão de 1,0**: das 15 vezes em que prevemos "conta paga", acertamos as 15. **Revocação de 0,682**: dos 22 usuários que de fato pagam, encontramos 15 — deixamos sete passar. É um modelo **conservador**: ele só aponta o dedo quando tem certeza, e o preço disso é perder um terço dos pagantes. Não escolhemos isso; caiu assim. A seção seguinte mostra que a escolha existe e onde ela é feita.

> **⚠️ Atenção — A acurácia, de novo, é a métrica que menos diz**
>
> A acurácia deu 0,894, e é o segundo número mais alto dos quatro — atrás de uma precisão de 1,0 que, sozinha, também não diz nada (um modelo que apontasse **um** pagante e acertasse teria precisão 1,0 igual). É a luz que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) instalou, acendendo duas vezes.
>
> Só 22 dos 66 usuários do conjunto de teste pagam. Um classificador que respondesse "não paga" para todo mundo, sem olhar dado nenhum, acertaria 44 dos 66 — **acurácia de 0,667**, com revocação exatamente zero e precisão indefinida (o mesmo `ZeroDivisionError` que o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html) exibiu). Dois terços de acerto de graça, por olhar dado nenhum. E não é hipótese: é exatamente o modelo que a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) obteve quando esqueceu de reescalonar — o mesmo classificador trivial, que lá acertou 148 dos 200, com coeficientes de aparência respeitável. A taxa de acerto não é a mesma nos dois lugares — 0,74 sobre a base inteira, 0,667 aqui —, e a diferença é a do parágrafo acima: este conjunto de teste ficou com 22 pagantes em 66, um terço, contra os 26% da base.
>
> O que separa o nosso modelo daquele não aparece na acurácia — 0,894 contra 0,667, uma diferença que parece pequena para a distância entre "um modelo" e "nenhum". Aparece na revocação: 0,682 contra 0,0.

### O limiar de 0,5 é uma escolha, não uma propriedade

A comparação `previsoes >= 0.5` está no meio daquele chunk como se fosse parte do algoritmo. Não é. O modelo produz probabilidades; **0,5 é uma decisão de quem usa o modelo**, e trocá-la troca todas as métricas:

In [ ]:
print(f"{'limiar':>7} {'VP':>4} {'FP':>4} {'FN':>4} {'VN':>4} {'precisão':>10} {'revocação':>11}")

for limiar in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    previsto = previsoes >= limiar
    tp = np.sum(previsto & real)
    fp = np.sum(previsto & ~real)
    fn = np.sum(~previsto & real)
    tn = np.sum(~previsto & ~real)
    print(f"{limiar:>7.2f} {tp:>4} {fp:>4} {fn:>4} {tn:>4} "
          f"{tp / (tp + fp):>10.4f} {tp / (tp + fn):>11.4f}")

> **🔷 Conceito**
>
> A tabela é o compromisso da [seção 8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) instanciado num modelo real. Baixar o limiar para 0,05 leva a revocação a 0,955 — o modelo passa a encontrar 21 dos 22 pagantes — ao custo de derrubar a precisão para 0,656, porque agora ele acusa 11 pessoas que não pagam. Subir para 0,9 inverte tudo: precisão de 1,0, revocação de 0,227, cinco pagantes encontrados de 22.
>
> **O modelo é o mesmo em todas as linhas.** O `beta` não mudou; nem um epoch de gradiente descendente foi rodado entre uma linha e a seguinte. O que muda é onde se corta a probabilidade, e essa escolha depende do custo de cada tipo de erro — que vem do problema, não dos dados. Se o objetivo é uma campanha de marketing barata para converter usuários, um falso positivo custa um e-mail e vale a pena baixar o limiar. Se o objetivo é dar desconto para quem já ia pagar, cada falso positivo é dinheiro perdido, e o limiar sobe.
>
> Repare também no que a tabela **não** faz: entre 0,5 e 0,9 a precisão não se mexe. Não é estabilidade do modelo; é que quase não há previsões naquela faixa — só nove das 66 caem entre 0,4 e 0,8. As probabilidades deste modelo se amontoam nas duas pontas, e um limiar só morde onde há pontos. Repare, de quebra, que a precisão nem sequer é monótona: ela cai de 0,905 para 0,895 entre 0,2 e 0,3, porque naquele intervalo o limiar perdeu dois verdadeiros positivos e nenhum falso positivo.

### Previsto contra real

Dá também para desenhar as previsões contra os valores reais:

In [ ]:
# Figura: Regressão logística: probabilidade prevista contra o resultado real
plt.scatter(previsoes, y_test, marker='+')
plt.axvline(0.5, color='gray', linestyle=':')
plt.xlabel("probabilidade prevista")
plt.ylabel("resultado real")
plt.yticks([0, 1])
plt.title("Regressão logística: previsto contra real")
plt.show()

Compare com a figura equivalente da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html), a do modelo linear. Duas diferenças importam. A primeira: aqui **todo ponto está entre 0 e 1** — não há mais probabilidade negativa, porque a função logística tornou isso impossível, não improvável. A segunda: as duas fileiras não se sobrepõem do mesmo jeito. Os usuários que não pagam ficam **todos** à esquerda da linha pontilhada, a maioria com probabilidade quase nula; os que pagam se espalham pelo gráfico inteiro. A separação não é limpa, e o que a estraga está de um lado só: **sete pagantes caem à esquerda da linha pontilhada**, e apenas um deles é limítrofe — os outros seis vão de 0,418 até 0,035, lá na ponta em que o modelo não hesitou, apenas errou. Perto do limiar ficam os casos difíceis; nas pontas ficam os erros caros.

In [ ]:
previsto = previsoes >= 0.5
erra = previsto != real

print(f"{np.sum(erra)} erros em {len(y_test)} pontos de teste")
for p, y in sorted(zip(previsoes[erra], y_test[erra])):
    print(f"   probabilidade prevista {p:.4f}, resultado real {y:.0f}")

Sete erros, e eles são **todos do mesmo tipo**: sete pagantes que o modelo deixou passar, e nenhum não pagante acusado por engano. Um é limítrofe — com 0,4920 o modelo quase acertou, e é o tipo de erro que se aceita, porque bastaria mover o limiar oito milésimos. Outros dois, com 0,4180 e 0,2515, são hesitações razoáveis: o modelo não tinha certeza e errou o lado. Os quatro restantes são apostas erradas com convicção crescente.

O último é diferente. É um usuário que **paga**, e a quem o modelo atribuiu probabilidade **0,0350** — ou seja, declarou com 96,5% de confiança que não pagaria. Esse não é um erro de calibração de limiar; nenhum limiar entre 0 e 1 o corrigiria sem destruir o resto. É um ponto que está do lado errado da fronteira que o modelo traçou, e nenhuma reta traçada naquele plano conseguiria colocá-lo do lado certo sem levar junto uma porção de não pagantes. Que fronteira é essa, e o que significa não existir uma que separe as classes perfeitamente, é o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html).

> **💡 Dica — Na prática: `scikit-learn`**
>
> Tudo o que esta seção calculou à mão cabe em três linhas:
>
> ```python
> from sklearn.metrics import classification_report, confusion_matrix
>
> previsto = (logistic(x_test @ beta) >= 0.5).astype(int)
> print(confusion_matrix(y_test, previsto))
> print(classification_report(y_test, previsto))
> ```
>
> `classification_report` devolve precisão, revocação e F1 **para cada classe**, e não só para a classe positiva — o que é uma correção útil ao hábito de olhar só o lado que interessa.
>
> Sobre o limiar, há uma armadilha de interface que vale conhecer. Um modelo do `scikit-learn` tem dois métodos:
>
> - `modelo.predict(X)` devolve rótulos, e embute o limiar de 0,5 sem perguntar;
> - `modelo.predict_proba(X)` devolve as probabilidades, e deixa a decisão com você.
>
> Quem usa só o `predict` nunca vê que houve uma escolha — e é exatamente a escolha que esta seção mostrou valer a diferença entre revocação 0,23 e 0,95. Para explorar o compromisso inteiro de uma vez, `sklearn.metrics.precision_recall_curve` percorre todos os limiares possíveis, e `roc_curve` faz o mesmo do ponto de vista da taxa de falsos positivos.

## Máquinas de Vetores de Suporte

> **📌 Nota**
>
> Esta seção corresponde a *Support Vector Machines*, do capítulo 16 de Grus (2019).

O conjunto de pontos onde `x_i @ beta` vale exatamente 0 é a **fronteira** entre as nossas duas classes: ali a função logística devolve 0,5, e a decisão vira cara ou coroa. De um lado prevemos "paga", do outro "não paga". Vale desenhar essa fronteira para ver com clareza o que o modelo está fazendo.

Os coeficientes abaixo são o `beta_unscaled` da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) — o mesmo ajuste, com a escala desfeita no fim, medido em anos e em reais. É a forma de que precisamos para desenhar a fronteira no plano do gráfico.

In [ ]:
import numpy as np
from typing import Tuple
from matplotlib import pyplot as plt

from scratch_np.logistic_regression import (
    xs, ys, logistic, negative_log_likelihood, negative_log_gradient,
)
from scratch_np.gradient_descent import gradient_step
from scratch_np.machine_learning import train_test_split

def scale(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    return X.mean(axis=0), X.std(axis=0, ddof=1)

def rescale(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    means, stdevs = scale(X)
    rescaled = X.copy()
    varia = stdevs > 0
    rescaled[:, varia] = (X[:, varia] - means[varia]) / stdevs[varia]
    return rescaled

rescaled_xs = rescale(xs)

rng = np.random.default_rng(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33, rng)

beta = rng.random(3)
for _ in range(5000):
    beta = gradient_step(beta, negative_log_gradient(x_train, y_train, beta), -0.01)

means, stdevs = scale(xs)
beta_unscaled = np.array([beta[0]
                          - beta[1] * means[1] / stdevs[1]
                          - beta[2] * means[2] / stdevs[2],
                          beta[1] / stdevs[1],
                          beta[2] / stdevs[2]])

beta_unscaled

A fronteira é `beta_unscaled[0] + beta_unscaled[1] * experiência + beta_unscaled[2] * salário = 0`. Isolando o salário, ela vira uma reta no plano do gráfico da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) — e `salario_na_fronteira` recebe a `grade` inteira de uma vez, em vez de ser chamada uma vez por ponto:

In [ ]:
# Figura: Usuários pagantes e não pagantes, com a fronteira de decisão
b0, b1, b2 = beta_unscaled

def salario_na_fronteira(experiencia):
    return -(b0 + b1 * experiencia) / b2

paga = ys == 1

plt.scatter(xs[paga, 1], xs[paga, 2], marker='+', label='paga')
plt.scatter(xs[~paga, 1], xs[~paga, 2], marker='.', label='não paga')

grade = np.array([0.0, 10.0])
plt.plot(grade, salario_na_fronteira(grade),
         color='black', label='fronteira de decisão')

plt.xlabel("anos de experiência")
plt.ylabel("salário")
plt.legend(loc='upper left')
plt.title("Fronteira de decisão")
plt.show()

`xs @ beta_unscaled >= 0` é a decisão do modelo para os 200 usuários de uma vez, e compará-la com `ys == 1` dá a máscara dos erros:

In [ ]:
errados = np.sum((xs @ beta_unscaled >= 0) != (ys == 1))

print(f"{errados} dos {len(xs)} usuários ficam do lado errado da fronteira")

A reta atravessa a nuvem na diagonal, subindo da esquerda para a direita: abaixo dela quase todo mundo paga — 33 dos 38 usuários que caem ali —, e acima dela quase todo mundo não paga: 143 dos 162. É a região de "experiência alta, salário baixo" que a seção 13.1 já tinha identificado a olho nu, agora com um contorno. Vinte e quatro dos 200 usuários ficam do lado errado dela — e agora, ao contrário do conjunto de teste da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html), há erros dos dois tipos: 19 pagantes acima da reta e cinco não pagantes abaixo dela.

> **🔷 Conceito**
>
> Essa fronteira é um **hiperplano** que corta o **espaço de atributos** em dois semiespaços — em duas dimensões, uma reta; em três, um plano; em mais, algo que não dá para desenhar mas que se comporta igual.
>
> E note como ela apareceu: **de brinde**. Nós não pedimos uma fronteira; pedimos o $\beta$ que maximiza a verossimilhança dos dados sob o modelo logístico. O hiperplano é um efeito colateral desse pedido.
>
> A pergunta que abre esta seção é: e se a fronteira fosse o pedido principal?

> **⚠️ Atenção — Espaço de atributos, não espaço de parâmetros**
>
> Neste ponto é fácil trocar um espaço pelo outro, porque os dois têm três dimensões neste problema. Vale separá-los com cuidado.
>
> - O **espaço de atributos** é onde os dados moram: cada ponto é um usuário, $(1, \text{experiência}, \text{salário})$. É o plano da figura logo acima. A fronteira é o conjunto $\{\mathbf{x} : \beta \cdot \mathbf{x} = 0\}$ — um conjunto de **x**, portanto um objeto deste espaço.
> - O **espaço de parâmetros** é onde mora o $\beta$: cada ponto dele é um modelo candidato inteiro. Foi por ele que o gradiente descendente caminhou na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) — os 5.000 epochs são uma trajetória aqui, não ali —, e é sobre ele que se desenha a superfície de perda do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html).
>
> O que gera a confusão é uma simetria real: $\beta \cdot \mathbf{x} = 0$ define um hiperplano nos **dois** espaços, dependendo de qual dos dois vetores você segura fixo. Com $\beta$ fixo e $\mathbf{x}$ variando, sai a fronteira de decisão. Com um $\mathbf{x}$ fixo e $\beta$ variando, sai o conjunto de todos os modelos que consideram aquele usuário exatamente limítrofe — um objeto legítimo e útil, mas que não é o que está desenhado no gráfico.
>
> A troca é de vocabulário, e nem por isso é inofensiva: quem embaralha os dois espaços não tem como entender por que multiplicar $\beta$ por 2 muda o modelo sem mover um milímetro da fronteira. É precisamente o que acontece no callout mais abaixo, sobre dados separáveis.

### O critério da margem

A ideia da **máquina de vetores de suporte** (*support vector machine*, ou SVM) é essa. Em vez de estimar probabilidades e deixar a fronteira aparecer no fim, ela procura diretamente o hiperplano que **melhor separa** as classes nos dados de treino — e "melhor" tem uma definição precisa: o hiperplano que **maximiza a distância até o ponto mais próximo de cada classe**. Essa distância se chama **margem**, e os pontos que a tocam são os **vetores de suporte** que dão nome ao método.

Por que isso é um critério, e não só uma preferência estética? Porque, quando as classes são separáveis, existem **infinitos** hiperplanos que classificam o treino sem erro nenhum, e eles não são equivalentes num conjunto de teste:

In [ ]:
# Figura: Quatro retas que separam perfeitamente o mesmo treino; só uma maximiza a margem
classe_a = np.array([[-2.0, -1.0], [-1.0, -2.0], [-1.5, -1.5], [-2.5, -0.5], [-3.0, -1.8]])
classe_b = np.array([[2.0, 1.0], [1.0, 2.0], [1.5, 1.5], [2.5, 0.5], [3.0, 1.8]])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(classe_a[:, 0], classe_a[:, 1], marker='.', s=90)
ax.scatter(classe_b[:, 0], classe_b[:, 1], marker='+', s=90)

grade = np.array([-4.0, 4.0])

# três separadoras válidas, mas desnecessariamente próximas de algum ponto
for a, b in [(-1.0, -2.0), (-1.0, 2.0), (-0.5, -1.0)]:
    ax.plot(grade, a * grade + b, ':', color='gray')

# a de margem máxima, com as duas linhas que a margem toca
ax.plot(grade, -grade, '-', color='black', linewidth=2)
ax.plot(grade, -grade - 3, '--', color='black', linewidth=0.8)
ax.plot(grade, -grade + 3, '--', color='black', linewidth=0.8)

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_title("Qual das retas separadoras é a melhor?")
plt.show()

As quatro retas acertam **todos** os pontos de treino: as três pontilhadas cinzas e a cheia preta classificam este conjunto sem um erro sequer. A diferença está na folga. A reta cheia é a que fica mais longe dos pontos mais próximos de cada lado, e as duas tracejadas finas marcam onde a margem dela encosta — nos quatro pontos de cada classe que são, por definição, os vetores de suporte. As pontilhadas passam a cerca de um terço dessa distância de algum ponto de treino.

A intuição é que um ponto novo, que caia um pouco fora do padrão do treino, tem mais chance de continuar do lado certo se a fronteira estiver o mais afastada possível de todo mundo.

> **❗ Importante — O critério da margem não é o critério da verossimilhança**
>
> Esta é a diferença que importa entre os dois métodos, e ela tem duas faces.
>
> **A primeira: quais pontos importam.** A log-verossimilhança negativa que a [seção 13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) construiu soma uma parcela por ponto — **todos** os pontos, inclusive os que estão longe e do lado certo. Um usuário para quem o modelo prevê probabilidade 0,999 e que de fato paga ainda contribui com uma perda pequena, mas positiva, e o gradiente ainda tem um termo para ele. Mover esse usuário para ainda mais longe da fronteira **melhora** a verossimilhança e desloca o $\beta$ ajustado. Já a margem máxima depende **apenas** dos pontos mais próximos: mover um ponto distante não muda o hiperplano da SVM em coisa nenhuma, porque ele não é um vetor de suporte.
>
> **A segunda: em dados separáveis, a verossimilhança não tem máximo.** Este é o caso em que a diferença deixa de ser filosófica. Se existe um hiperplano que separa as classes perfeitamente, então multiplicar $\beta$ por 2 empurra todas as probabilidades corretas ainda mais para perto de 0 ou 1 e **reduz** a log-verossimilhança negativa. Multiplicar por 4 reduz mais. Não existe um $\beta$ ótimo: existe uma direção ótima e um comprimento que cresce para sempre.
>
> Dá para ver isso acontecendo — nos mesmos pontos da figura acima, menos o mais afastado de cada classe, para caber em duas linhas de código:

In [ ]:
xs_sep = np.array([[1.0, -2.0, -1.0], [1.0, -1.0, -2.0], [1.0, -1.5, -1.5], [1.0, -2.5, -0.5],
                   [1.0,  2.0,  1.0], [1.0,  1.0,  2.0], [1.0,  1.5,  1.5], [1.0,  2.5,  0.5]])
ys_sep = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0])

rng_sep = np.random.default_rng(0)
beta_sep = rng_sep.random(3)
marcos = [1, 10, 100, 1000, 10000, 50000]

for epoch in range(1, max(marcos) + 1):
    gradiente = negative_log_gradient(xs_sep, ys_sep, beta_sep)
    beta_sep = gradient_step(beta_sep, gradiente, -0.01)
    if epoch in marcos:
        norma = np.linalg.norm(beta_sep)
        perda = negative_log_likelihood(xs_sep, ys_sep, beta_sep)
        print(f"epoch {epoch:>6}: |beta| = {norma:8.4f}   perda = {perda:.6f}")

> `np.linalg.norm` é o comprimento do vetor — a função `magnitude` do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html), que era `math.sqrt(sum_of_squares(v))`, numa chamada só —, e o `rng_sep` é um gerador próprio porque este conjunto é inventado aqui e não tem nada a ver com a divisão treino/teste do resto do capítulo.
>
> O comprimento de $\beta$ cresce sem parar e a perda desce em direção a zero sem nunca chegar. O gradiente descendente não "converge" aqui — ele apenas anda cada vez mais devagar, e para onde estava quando você mandou parar. **O modelo que você obtém é uma função de quantos epochs você teve paciência de rodar**, o que é uma forma desconfortável de escolher um modelo.
>
> É por isso que o `LogisticRegression` do `scikit-learn` liga a regularização de fábrica, como o callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) mencionou: a penalidade sobre o tamanho de $\beta$ é o que impede essa fuga para o infinito. A máquina de vetores de suporte não precisa desse remendo — o critério de margem já é uma restrição sobre o tamanho do vetor, embutida na definição do problema.

### Quando não existe hiperplano separador

Encontrar o hiperplano de margem máxima é um problema de otimização com restrições, e as técnicas que o resolvem estão fora do alcance deste livro — é por isso que esta seção não implementa a máquina de vetores de suporte, e é a única do capítulo que deixa de implementar o seu próprio assunto.

Mas há um problema anterior e mais interessante: **um hiperplano separador pode simplesmente não existir**. É o caso destes dados. Não é uma impressão vinda do gráfico; dá para verificar por busca exaustiva.

In [ ]:
def separavel(pontos: np.ndarray, rotulos: np.ndarray) -> bool:
    """Existe alguma reta que separe perfeitamente as duas classes?"""
    i, j = np.triu_indices(len(pontos), k=1)
    segmentos = pontos[j] - pontos[i]                 # a direção de cada par

    # a direção do segmento, e a perpendicular a ela
    direcoes = np.vstack([segmentos,
                          np.column_stack([segmentos[:, 1], -segmentos[:, 0]])])
    direcoes = direcoes[np.any(direcoes != 0, axis=1)]

    proj0 = direcoes @ pontos[rotulos == 0].T         # (direções, não pagantes)
    proj1 = direcoes @ pontos[rotulos == 1].T         # (direções, pagantes)

    return bool(np.any((proj0.max(axis=1) < proj1.min(axis=1)) |
                       (proj1.max(axis=1) < proj0.min(axis=1))))

pontos = xs[:, 1:]

if separavel(pontos, ys):
    print("Existe uma reta que separa pagantes de não pagantes.")
else:
    print("Nenhuma reta separa estes dados.")

Vale ver o que aconteceu com os dois laços aninhados. `np.triu_indices` produz os índices dos 19.900 pares sem repetir nenhum, e `pontos[j] - pontos[i]` calcula as 19.900 direções de uma vez. `np.vstack` empilha um array sobre o outro, na vertical: aqui, as direções dos segmentos por cima das perpendiculares a elas, o que dobra a contagem antes de as nulas serem descartadas. `direcoes @ pontos.T` projeta **todos** os pontos sobre **todas** as direções numa multiplicação de matrizes: o resultado tem uma linha por direção e uma coluna por ponto, e a pergunta "as duas classes se separam nesta direção?" vira uma comparação entre os máximos e os mínimos de cada linha, com `axis=1`. Nenhum `for` sobrou — o que era um laço duplo sobre 200 pontos é uma conta de quase 40 mil por 200 que o `numpy` faz em milésimos de segundo.

> **📌 Nota**
>
> Por que testar só direções ligadas a **pares de pontos** basta, e a busca é exaustiva apesar de finita: se as duas classes são separáveis no plano, os fechos convexos delas são disjuntos, e o segmento mais curto entre os dois fechos define a direção de separação. Esse segmento ou liga dois vértices — um de cada classe, e a direção é a do próprio segmento — ou liga um vértice a uma aresta, e aí a direção é perpendicular a essa aresta, que por sua vez liga dois pontos da mesma classe. Percorrer todos os pares nas duas formas cobre os dois casos. São 19.900 pares e 39.794 direções — seriam 39.800, mas três pares de usuários têm exatamente a mesma experiência e o mesmo salário, e os seis vetores nulos que eles produzem saem da conta —, tudo em cerca de sete milésimos de segundo. A matriz de projeções ocupa cerca de 60 MB, que é o preço de fazer tudo de uma vez.

Não há reta nenhuma que separe pagantes de não pagantes neste conjunto. A SVM, na forma acima, não teria o que devolver.

### O truque do kernel

Às vezes dá para contornar isso **transformando os dados num espaço de dimensão maior**. Considere o conjunto unidimensional mais simples possível em que nenhum ponto de corte funciona:

In [ ]:
# Figura: À esquerda, um conjunto unidimensional não separável; à direita, o mesmo conjunto depois do mapa x → (x, x²)
valores = np.array([-3.0, -2.5, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 2.5, 3.0])
rotulos = (np.abs(valores) >= 2).astype(int)

fig, (esquerda, direita) = plt.subplots(1, 2, figsize=(10, 4))

for marcador, classe in [('+', 1), ('.', 0)]:
    selecionados = valores[rotulos == classe]
    esquerda.scatter(selecionados, np.zeros_like(selecionados), marker=marcador, s=90)
    direita.scatter(selecionados, selecionados ** 2, marker=marcador, s=90)

esquerda.set_yticks([])
esquerda.set_ylim(-1, 1)
esquerda.set_box_aspect(0.3)   # o painel da esquerda é uma reta, não um plano
esquerda.set_anchor("N")
esquerda.set_xlabel("x")
esquerda.set_title("Uma dimensão: nenhum corte separa")

direita.axhline(2.5, color='black')
direita.set_xlabel("x")
direita.set_ylabel("x²")
direita.set_title("Duas dimensões: uma reta separa")

plt.tight_layout()
plt.show()

À esquerda, uma classe ocupa as duas pontas da reta e a outra ocupa o meio. Não existe ponto de corte que resolva: qualquer corte único deixa exemplos da classe das pontas dos dois lados. À direita, o mesmo conjunto depois de enviar cada ponto $x$ para o par $(x, x^2)$. Nada foi inventado — a segunda coordenada é uma função da primeira, calculada a partir dela —, mas agora a reta horizontal $y = 2{,}5$ separa perfeitamente.

> **🔷 Conceito**
>
> Isso normalmente é chamado de **truque do kernel**, e o "truque" está numa economia. Mapear de fato todos os pontos para o espaço de dimensão maior pode ser caro — ou impossível, se o espaço tiver dimensão infinita, o que acontece com alguns kernels usados na prática.
>
> Acontece que o algoritmo da SVM só precisa dos **produtos escalares** entre pares de pontos, nunca das coordenadas individuais. Uma função de *kernel* calcula esses produtos escalares no espaço maior **direto a partir das coordenadas originais**, sem nunca construir os vetores transformados. Você ganha a fronteira curva sem pagar o custo da dimensão.
>
> É o que a SVM compra, e é a razão de ela ter dominado a classificação durante uma década, antes das redes neurais: **fronteiras não lineares com um problema de otimização que continua convexo** — logo, com solução única, sem mínimos locais e sem dependência do ponto de partida. Compare com o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html), onde a não linearidade vem de outro jeito e custa exatamente essas garantias.

Usar máquinas de vetores de suporte sem depender de software de otimização especializado, escrito por quem tem o preparo para isso, é difícil e provavelmente não é uma boa ideia — e é aqui que Grus (2019) encerra o tratamento do assunto. Este livro faz o mesmo: esta é a única seção do capítulo sem uma implementação do zero, e a razão não é falta de espaço. É que a implementação honesta seria um algoritmo de programação quadrática com restrições, e escrever um não ensinaria nada sobre classificação.

### O que este capítulo deixa

Três capítulos, uma escada, e o degrau de cima é este. A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) ajustou uma reta com uma **fórmula fechada** — duas médias, uma covariância, uma variância, sem laço nenhum. O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) manteve a fórmula fechada existindo e parou de usá-la: a solução exata era um sistema linear que aquele capítulo escolheu não resolver, por escopo. Aqui ela não existe, e a promessa da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) foi cobrada — o gradiente descendente deixou de ser conveniência e virou a única via até um modelo. É assim também nos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) — mas não em todo o resto: os capítulos [14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) e [17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html) constroem modelos que não descem gradiente nenhum, como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) já avisava.

Há uma segunda herança, e ela só ficou visível na figura desta seção. Os três modelos da escada, mais a máquina de vetores de suporte, decidem todos comparando `x @ beta` com um limiar — e produto escalar contra limiar é, sempre, um **hiperplano no espaço de atributos**. Uma reta, no nosso caso. Os 24 usuários do lado errado dela não estão lá por falta de epochs de treino: estão lá porque nenhuma reta os colocaria do lado certo. O truque do kernel é uma saída para isso, e uma saída cara — ele compra a curvatura trocando o espaço, sem abandonar o produto escalar, e paga com um custo que cresce entre o quadrado e o cubo do número de pontos, e com uma fronteira que ninguém mais consegue ler.

O [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) chega à curvatura por um caminho que não tem nada disso. A árvore de decisão não tem produto escalar, não tem coeficiente, não tem gradiente e não tem ponto de partida aleatório; ela pergunta uma coisa de cada vez — "salário acima de 60.000?" — e a fronteira que sai é feita de degraus paralelos aos eixos. Uma consequência disso fecha justamente a lição da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html): para uma árvore, **medir salário em reais ou em desvios padrão é rigorosamente indiferente**. Reescalonar não muda uma única pergunta que ela faça, porque toda pergunta é sobre a **ordem** dos valores de uma coluna, e reescalonar preserva a ordem. A perda `nan` da primeira conta, a saturação em `float64`, a taxa de aprendizado que serve para uma coluna e não para a outra — nada disso existe no próximo capítulo. Some o problema inteiro, junto com o produto escalar que o criava.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O `scikit-learn` traz máquinas de vetores de suporte em `sklearn.svm`, e a escolha entre as classes importa:
>
> ```python
> from sklearn.svm import SVC, LinearSVC
>
> modelo = SVC(kernel='rbf', C=1.0)      # com kernel; fronteira não linear
> modelo = LinearSVC(C=1.0)              # sem kernel; muito mais rápido
> ```
>
> `SVC` é a implementação com kernel, e por baixo dela roda o **LIBSVM**, exatamente o software especializado que a citação acima diz que você deveria usar. O `kernel='rbf'` (*radial basis function*) é o padrão e corresponde a um espaço de dimensão infinita — o caso extremo em que mapear os pontos de verdade seria literalmente impossível e só o truque do kernel torna a conta viável.
>
> Três coisas que valem a pena saber:
>
> - **`C` é o mesmo tipo de hiperparâmetro da regularização** da [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html). Como dados reais quase nunca são separáveis — os nossos não são —, a formulação usada na prática é a de *margem suave*, que permite alguns pontos violarem a margem e cobra por isso. `C` é quanto se cobra.
> - **Reescalonar não é opcional.** A margem é medida com distância euclidiana, então uma coluna em reais e outra em anos produzem o mesmo desastre que a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) mostrou — aqui sem nenhum `nan` na tela, só com um modelo silenciosamente dominado pela coluna de escala maior. O guia prático do LIBSVM começa por essa recomendação.
> - **`SVC` não escala para dados grandes.** O custo do treino cresce entre o quadrado e o cubo do número de pontos, o que na prática limita `SVC` a algumas dezenas de milhares de exemplos. Acima disso usa-se `LinearSVC` (sem kernel) ou `SGDClassifier`, que aplica a mesma perda de margem por gradiente descendente estocástico — o método do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), de volta pela última vez neste capítulo.
>
> E vale registrar a diferença de interface que resume a diferença de critério: `SVC` **não tem** `predict_proba` habilitado por padrão. Ele devolve o lado do hiperplano, não uma probabilidade, porque nunca modelou uma. Obter probabilidades dele exige uma calibração extra por cima, e o `scikit-learn` acabou de mudar como se pede isso: o parâmetro `SVC(probability=True)` foi **depreciado na versão 1.9** e será removido na 1.11, em favor de
>
> ```python
> from sklearn.calibration import CalibratedClassifierCV
>
> modelo = CalibratedClassifierCV(SVC(), ensemble=False)
> ```
>
> A troca melhora o nome, porque agora a classe diz o que faz — antes, um parâmetro booleano escondia um segundo modelo ajustado por baixo. E a ironia sobrevive intacta: essa calibração ajusta **uma regressão logística** sobre as saídas da SVM. O método que este capítulo construiu volta no fim como remendo, para dar à máquina de vetores de suporte a probabilidade que o critério de margem se recusou a modelar.

## Leituras adicionais

A seção "For Further Investigation" do capítulo 16 de Grus (2019) faz duas sugestões.

A primeira é o `scikit-learn`, que tem módulos tanto para [regressão logística](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) quanto para [máquinas de vetores de suporte](https://scikit-learn.org/stable/modules/svm.html). Os callouts de fechamento de cada seção deste capítulo mostram as duas interfaces.

A segunda é o [LIBSVM](https://www.csie.ntu.edu.tw/~cjlin/libsvm/), a implementação de máquinas de vetores de suporte que o `scikit-learn` usa por baixo em `SVC` e `SVR`. O site traz um guia prático curto — *A Practical Guide to Support Vector Classification* — que vale a leitura justamente por ser prático: ele começa recomendando reescalonar os dados, que é a lição que a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) aprende do jeito difícil.

Para o tratamento estatístico da regressão logística — testes sobre coeficientes, razões de chances, diagnóstico —, James et al. (2021) é o ponto de partida usual, e Hastie et al. (2009) vai fundo tanto na logística quanto no problema de otimização que a máquina de vetores de suporte resolve e que este capítulo não implementa.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani**. *An Introduction to Statistical Learning*. 2nd ed.. Springer. 2021.